# **Preparación de datos: división entrenamiento / prueba**

En este notebook se prepara el dataset de ingreso de viajeros para el modelado de series de tiempo, buscando dejarlo separado en un conjunto de entrenamiento y uno de prueba bajo una distribución 70/30. Cabe mencionar que, al tratarse de series de tiempo, la división no se hace de forma aleatoria sino respetando la continuidad temporal de las observaciones, de tal forma que se mantenga la linealidad de la serie y no se rompa el orden cronológico, ya que ese orden es justamente lo que se quiere modelar. Finalmente, el resultado se exporta a `datos/processed/entrenamiento.csv` y `datos/processed/prueba.csv`.

## **Carga y limpieza**

Primero se carga la hoja `Datos` y se aplican únicamente las transformaciones que consideramos necesarias, buscando no alterar de más la información original. Se elimina la columna `Mes` (texto) por ser redundante con `Mes cod`, ya que el mes ya lo tenemos en su formato numérico y no aporta nada adicional tenerlo dos veces; asimismo, se agrega una columna `Fecha` (primer día de cada mes) que facilita bastante el trabajo con series de tiempo, y una columna indicadora `pandemia` que marca los meses fuertemente afectados por la pandemia (de marzo de 2020 a diciembre de 2021), esto para poder aportar al análisis posterior, dado que en el EDA inicial ya habíamos identificado comportamientos atípicos esperables en ese período.

**Filtro de categorías consistentes (Turista + Excursionista).** Cabe mencionar que entre 2022 y 2023 el tipo de viajero cambió de criterio, ya que se excluyeron los viajeros no turísticos de alta frecuencia, es decir, el comercio fronterizo y el tránsito. Es por esto que la categoría "Viajero" disminuye fuertemente en 2023 y el total anual parece caer, pero no se trata de una caída real de turismo sino de un cambio metodológico. Bajo esta idea, y para poder comparar de forma consistente en todo el período, nos quedamos únicamente con Turista + Excursionista, que sí mantienen la misma definición a lo largo de toda la serie. Adicional, cabe mencionar que este filtro también deja fuera la categoría "Cruceristas" (que solo está disponible hasta 2022) y, de paso, elimina un valor contaminante ("Cruceros") que se colaba en la columna `Frontera`.

**Nota sobre la vía Marítima.** Por otro lado, la vía marítima pierde detalle de registro desde 2017 y, tras aplicar el filtro anterior, queda con bastantes meses sin observaciones (96 de 147 meses en entrenamiento). Es por esto que su serie mensual necesitará reindexar al rango de fechas completo y rellenar los meses sin arribos al momento de construirla, lo cual haremos en el notebook de series por vía y no en esta etapa de limpieza, ya que en este formato largo no tendríamos con qué completar esas filas.

In [1]:
import os
import pandas as pd

ruta_dataset = "../../datos/raw/Base_Migracion_2009-2026jun.xlsx"
viajeros = pd.read_excel(ruta_dataset, sheet_name="Datos")

# Columna Fecha (primer día de cada mes) a partir de Año y Mes cod.
viajeros["Fecha"] = pd.to_datetime(
    dict(year=viajeros["Año"], month=viajeros["Mes cod"], day=1)
)

# Se elimina el nombre del mes en texto (redundante con Mes cod).
viajeros = viajeros.drop(columns=["Mes"])

# Se conservan únicamente las categorías consistentes en todo el período: Turista y Excursionista.
# Esto excluye el tipo "Viajero" (afectado por el quiebre metodológico 2022->2023) y "Cruceristas"
# (solo disponible hasta 2022), y elimina de paso el valor contaminante "Cruceros" de la columna Frontera.
viajeros = viajeros[viajeros["Tipo de Viajero"].isin(["Turista", "Excursionista"])].copy()

# Indicador de pandemia: meses fuertemente afectados (mar-2020 a dic-2021).
inicio_pandemia = pd.Timestamp("2020-03-01")
fin_pandemia = pd.Timestamp("2021-12-01")
viajeros["pandemia"] = viajeros["Fecha"].between(inicio_pandemia, fin_pandemia)

# Se reordena para dejar Fecha al inicio.
viajeros = viajeros[["Fecha", "Año", "Mes cod", "pandemia"] +
                    [c for c in viajeros.columns if c not in ["Fecha", "Año", "Mes cod", "pandemia"]]]
viajeros.head()

,Fecha,Año,Mes cod,pandemia,Vía,Frontera,País,Región,Región dos,Regiones OMT,MCEO,Agrupación Residencia,Tipo de Viajero,Viajero
0,2009-01-01,2009,1,False,Aérea,01 La Aurora,Albania,OTROS EUROPEOS,Europa,EUROPA MERIDIONAL,08 OTROS,Europa,Turista,1.0
1,2009-01-01,2009,1,False,Aérea,01 La Aurora,Alemania,EUROPA,Europa,EUROPA OCCIDENTAL,04 EUROPA,Europa,Turista,716.0
2,2009-01-01,2009,1,False,Aérea,01 La Aurora,Angola,OTROS PAISES DEL MUNDO,Otros Paises Del Mundo,ÁFRICA CENTRAL,08 OTROS,Resto del Mundo,Turista,4.0
3,2009-01-01,2009,1,False,Aérea,01 La Aurora,Arabia Saudita,OTROS ORIENTE MEDIO,Oriente Medio,ORIENTE MEDIO,08 OTROS,Resto del Mundo,Turista,5.0
4,2009-01-01,2009,1,False,Aérea,01 La Aurora,Argelia,OTROS PAISES DEL MUNDO,Otros Paises Del Mundo,ÁFRICA DEL NORTE,08 OTROS,Resto del Mundo,Turista,1.0


## **División 70/30**

El dataset cubre 210 meses consecutivos, que van de enero de 2009 a junio de 2026. La división 70/30 se aplica sobre el eje temporal, de tal forma que el 70% de los meses más antiguos conforma el conjunto de entrenamiento y el 30% más reciente el de prueba. Cabe mencionar que esta proporción es por tiempo, es decir por meses, y no por número de filas, ya que los años previos a 2023 tienen muchas más filas por mes (al venir desagregados por países individuales) que los años posteriores (que vienen por agrupaciones de mercado), de tal forma que el reparto de filas no queda exactamente en 70/30, pero lo que realmente interesa para una serie de tiempo es la cobertura temporal y no el conteo de filas.

In [2]:
# Corte temporal: 70% de los meses para entrenamiento, 30% para prueba.
meses_ordenados = sorted(viajeros["Fecha"].unique())
n_meses = len(meses_ordenados)
n_train = int(n_meses * 0.70)

fecha_corte_train = pd.Timestamp(meses_ordenados[n_train - 1])   # último mes de entrenamiento
fecha_inicio_prueba = pd.Timestamp(meses_ordenados[n_train])     # primer mes de prueba
fecha_inicio = pd.Timestamp(meses_ordenados[0])
fecha_fin = pd.Timestamp(meses_ordenados[-1])

pct_train = n_train / n_meses * 100
pct_prueba = (n_meses - n_train) / n_meses * 100

print(f"Total de meses: {n_meses}")
print(f"Entrenamiento: {n_train} meses ({pct_train:.1f}%) -> "
      f"de {fecha_inicio:%Y-%m} a {fecha_corte_train:%Y-%m}")
print(f"Prueba       : {n_meses - n_train} meses ({pct_prueba:.1f}%) -> "
      f"de {fecha_inicio_prueba:%Y-%m} a {fecha_fin:%Y-%m}")

Total de meses: 210
Entrenamiento: 147 meses (70.0%) -> de 2009-01 a 2021-03
Prueba       : 63 meses (30.0%) -> de 2021-04 a 2026-06


In [3]:
# Se separan los conjuntos respetando el orden temporal.
entrenamiento = viajeros[viajeros["Fecha"] <= fecha_corte_train].copy()
prueba = viajeros[viajeros["Fecha"] > fecha_corte_train].copy()

print(f"Filas entrenamiento: {len(entrenamiento):,} ({len(entrenamiento) / len(viajeros) * 100:.1f}%)")
print(f"Filas prueba       : {len(prueba):,} ({len(prueba) / len(viajeros) * 100:.1f}%)")
print(f"Meses de pandemia en entrenamiento: {entrenamiento.loc[entrenamiento['pandemia'], 'Fecha'].nunique()}")
print(f"Meses de pandemia en prueba       : {prueba.loc[prueba['pandemia'], 'Fecha'].nunique()}")

Filas entrenamiento: 116,966 (85.0%)
Filas prueba       : 20,676 (15.0%)
Meses de pandemia en entrenamiento: 13
Meses de pandemia en prueba       : 9


In [4]:
# Exportación a datos/processed/.
carpeta_salida = "../../datos/processed"
os.makedirs(carpeta_salida, exist_ok=True)

entrenamiento.to_csv(f"{carpeta_salida}/entrenamiento.csv", index=False)
prueba.to_csv(f"{carpeta_salida}/prueba.csv", index=False)
print("Archivos escritos:")
print(f"  {carpeta_salida}/entrenamiento.csv")
print(f"  {carpeta_salida}/prueba.csv")

Archivos escritos:
  ../../datos/processed/entrenamiento.csv
  ../../datos/processed/prueba.csv
